# 01 · Corpus de falacias: FallacyES y FallacyES-Political

Primer notebook del preprocesado (E2). Aquí se descargan y se exploran los tres corpus de falacias en español que usa el proyecto:

| Corpus | Qué contiene | Fuente |
|---|---|---|
| FallacyES — prototípicas | Falacias del corpus *Logic* traducidas al español (DeepL + revisión manual) y, para muchas, una versión sin falacia escrita a mano | idUS (Univ. de Sevilla), [DOI 10.12795/11441/167050](https://doi.org/10.12795/11441/167050) |
| FallacyES — espontáneas | Fragmentos de comentarios de meneame.net, con y sin falacia | idUS (Univ. de Sevilla), [DOI 10.12795/11441/167050](https://doi.org/10.12795/11441/167050) |
| FallacyES-Political | Falacias de 19 debates electorales españoles (1993–2023), transcritos con WhisperX | Zenodo, [DOI 10.5281/zenodo.14836328](https://doi.org/10.5281/zenodo.14836328) |

Los tres tienen licencia CC BY 4.0, así que hay que citarlos:
- Cruz, F. L., Troyano, J. A., Enríquez, F. y Ortega, F. J. (2023). Detección y clasificación de falacias prototípicas y espontáneas en español. *Procesamiento del Lenguaje Natural*, 71, 53–62.
- Cruz, F. L., Enríquez, F., Ortega, F. J. y Troyano, J. A. (2025). FallacyES-Political: A multiclass dataset of fallacies in Spanish political debates. *Procesamiento del Lenguaje Natural*, 74, 127–138.

**Por qué estos corpus y no *logical-fallacy* (el original en inglés):** ya están en español, así que no hay que traducir; traen ejemplos sin falacia, que *logical-fallacy* no tiene; y FallacyES-Political es del mismo dominio que el proyecto (discurso político español hablado).

Este notebook solo descarga y explora los datos. Las etiquetas, la limpieza, los splits y la exportación a JSONL vienen después.

## 0. Preparación

In [1]:
from pathlib import Path

import pandas as pd
import requests

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_FALLACYES = Path("../data/raw/fallacyes")
CARPETA_POLITICAL = Path("../data/raw/fallacyes_political")

# Para ver los textos (casi) enteros al mostrar tablas.
pd.set_option("display.max_colwidth", 200)


def contar_palabras(textos):
    """Número de palabras de cada texto (separando por espacios)."""
    return textos.str.split().str.len()

## 1. Descarga

Los ficheros se guardan en `data/raw/`, que no se sube a GitHub: cada miembro del grupo los descarga ejecutando esta sección. Las direcciones apuntan a versiones fijas (el registro v1.0.0 de Zenodo y ficheros concretos de idUS), así que todos trabajamos con los mismos datos.

Además de los datos se bajan la documentación de FallacyES (`FallacyES_readme.txt`), la guía de anotación de FallacyES-Political (`guia-anotacion.md`) y la lista de debates (`debates_info.csv`).

In [2]:
# Dónde se guarda cada fichero → de dónde se descarga.
DESCARGAS = {
    CARPETA_FALLACYES / "FallacyES-prototypical.csv":
        "https://idus.us.es/server/api/core/bitstreams/1a365d67-2543-47a4-9be3-e7b8ec2cec8f/content",
    CARPETA_FALLACYES / "FallacyES-spontaneous.csv":
        "https://idus.us.es/server/api/core/bitstreams/71a0a09c-f559-4637-b008-35ad5a6e765e/content",
    CARPETA_FALLACYES / "FallacyES_readme.txt":
        "https://idus.us.es/server/api/core/bitstreams/7878d236-7122-4749-9669-64da71a23e08/content",
    CARPETA_POLITICAL / "fallacyES-political.csv":
        "https://zenodo.org/records/14836328/files/fallacyES-political.csv?download=1",
    CARPETA_POLITICAL / "debates_info.csv":
        "https://zenodo.org/records/14836328/files/debates_info.csv?download=1",
    CARPETA_POLITICAL / "guia-anotacion.md":
        "https://zenodo.org/records/14836328/files/guia-anotacion.md?download=1",
}


def descargar(url, destino):
    """Descarga un fichero, salvo que ya exista."""
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return
    destino.parent.mkdir(parents=True, exist_ok=True)
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()  # se para con un error si la descarga falla
    destino.write_bytes(respuesta.content)
    print(f"Descargado: {destino.name} ({len(respuesta.content):,} bytes)")


for destino, url in DESCARGAS.items():
    descargar(url, destino)

Ya existe: FallacyES-prototypical.csv
Ya existe: FallacyES-spontaneous.csv
Ya existe: FallacyES_readme.txt
Ya existe: fallacyES-political.csv
Ya existe: debates_info.csv
Ya existe: guia-anotacion.md


## 2. FallacyES — falacias prototípicas

Cada fila es una falacia (`fallacy_example`) con su tipo (`label`) y, cuando los autores la escribieron, una versión corregida sin falacia (`non-fallacy_example`). Esas versiones sin falacia se crearon a mano modificando la falacia, así que no son textos reales.

In [3]:
proto = pd.read_csv(CARPETA_FALLACYES / "FallacyES-prototypical.csv")
print(proto.shape)
proto.head()

(2083, 3)


,label,fallacy_example,non-fallacy_example
0,faulty generalization,"""A Annie le debe gustar Starbucks porque a todas las chicas blancas les gusta Starbucks"".",A Annie le debe gustar Starbucks porque a ella le gustan las bebidas dulces y Starbucks tiene muchas opciones de bebidas dulces.
1,faulty generalization,"Este año hace más calor en Las Vegas que el año pasado; por lo tanto, el calentamiento global se está acelerando rápidamente.","El año pasado hacía más calor en Las Vegas que este año; por lo tanto, el calentamiento global se está ralentizando."
2,faulty generalization,"""Los dos cursos que tomé en la UF no fueron muy interesantes. No creo que sea una buena universidad"".",Los dos cursos que tomé en la UF me parecieron aburridos. No tengo una opinión formada sobre si la UF es una buena universidad o no.
3,faulty generalization,Un conductor con matrícula de Nueva York te corta el paso en el tráfico. Decides que todos los conductores de Nueva York son pésimos conductores.,Un conductor con matrícula de Nueva York te corta el paso en el tráfico. Esto no significa que todos los conductores de Nueva York sean pésimos conductores.
4,faulty generalization,¿Nadie en tu familia hace surf? Pero pensé que habías dicho que vivías en California antes de esto.,¿Nadie en tu familia hace surf? Pero tal vez vivieron en California en un momento de su vida y eso fue lo que te hizo pensar que lo hacían.


In [4]:
# Valores vacíos por columna
proto.isna().sum()

label                    0
fallacy_example          0
non-fallacy_example    765
dtype: int64

In [5]:
proto["label"].value_counts()

label
faulty generalization     399
ad hominem                258
ad populum                191
false causality           190
circular reasoning        158
deductive fallacy         141
appeal to emotion         140
fallacy of relevance      138
intentional               122
false dilemma             120
fallacy of credibility    118
fallacy of extension      108
Name: count, dtype: int64

In [6]:
# Longitud de las falacias en palabras
contar_palabras(proto["fallacy_example"]).describe()

count    2083.000000
mean       24.914066
std        17.068766
min         4.000000
25%        14.000000
50%        21.000000
75%        31.000000
max       193.000000
Name: fallacy_example, dtype: float64

In [7]:
# Textos repetidos
duplicadas = proto[proto["fallacy_example"].duplicated(keep=False)]
print("Filas con el texto repetido:", len(duplicadas))

# ¿Algún texto repetido tiene etiquetas distintas?
etiquetas_por_texto = duplicadas.groupby("fallacy_example")["label"].nunique()
print("Textos repetidos con etiquetas distintas:", (etiquetas_por_texto > 1).sum())

duplicadas.sort_values("fallacy_example").head(6)

Filas con el texto repetido: 48
Textos repetidos con etiquetas distintas: 1


,label,fallacy_example,non-fallacy_example
463,false causality,"""Ha llovido y hemos perdido el partido. Cada vez que llueve, vamos a perder"".","A veces llueve y perdemos el partido, pero no siempre es así."
484,false causality,"""Ha llovido y hemos perdido el partido. Cada vez que llueve, vamos a perder"".",No necesariamente vamos a perder cada vez que llueve.
1182,ad hominem,"Boris no está cualificado para hacer sugerencias sobre nuestro sistema penal. Como ex convicto, siempre se pondría del lado de los delincuentes.",NaN
1109,ad hominem,"Boris no está cualificado para hacer sugerencias sobre nuestro sistema penal. Como ex convicto, siempre se pondría del lado de los delincuentes.",NaN
414,false causality,"Cada vez que ese gallo canta, sale el sol. ¡Ese gallo debe ser muy poderoso e importante!","Cada vez que el gallo canta, es una señal de que se está acercando el amanecer. ¡Ese gallo debe tener un buen oído para saber cuándo llega el sol!"
581,false causality,"Cada vez que ese gallo canta, sale el sol. ¡Ese gallo debe ser muy poderoso e importante!",NaN


In [8]:
# Algunas falacias junto a su versión sin falacia
proto.dropna(subset=["non-fallacy_example"]).sample(5, random_state=0)

,label,fallacy_example,non-fallacy_example
1284,deductive fallacy,"Reconociendo que Amanda había cometido una falacia al argumentar que debíamos comer alimentos saludables porque un nutricionista decía que eran populares, Alyse dijo que, por lo tanto, debíamos co...","Reconociendo que Amanda había cometido una falacia al argumentar que debíamos comer alimentos saludables porque un nutricionista decía que eran populares, Alyse dijo que, por lo tanto, debíamos co..."
807,ad populum,"""Voy a comprar un iPhone porque todo el mundo tiene un iPhone y son geniales"".",Voy a comprar un iPhone porque me gusta el diseño y las características del teléfono.
1769,fallacy of relevance,"Señoras y señores del jurado, miren la ropa ensangrentada, el arma homicida. Imaginen los gritos de impotencia de la víctima. Un crimen así no merece otro veredicto que el de culpable, culpable.","El veredicto de un juicio debe basarse en pruebas y argumentos sólidos, en lugar de ser influenciado por emociones o apelaciones a la lógica."
486,false causality,"Lois y Jan hicieron un ""baile de la nieve"" una tarde, y esa noche nevó. Afirmaron haber traído la nieve.","Lois y Jan realizaron un ""baile de la nieve"" una tarde, pero no pueden afirmar haber traído la nieve."
1925,fallacy of credibility,"""Vamos a protestar y a no meternos en problemas porque el señor Iglesias ha dicho que está bien"".",Vamos a protestar pacíficamente y a no meternos en problemas porque creemos que es importante expresar nuestras opiniones.


## 3. FallacyES — falacias espontáneas

Cada fila es un fragmento de un comentario de meneame.net con su tipo (`label`, que vale `non-fallacy` si no hay falacia) y los datos de la noticia en la que se comentó. Los autores buscaron comentarios en los que un usuario acusaba a otro de una falacia y anotaron a mano el trozo falaz. Los textos sin falacia son fragmentos de otros comentarios de la misma noticia.

In [9]:
espont = pd.read_csv(CARPETA_FALLACYES / "FallacyES-spontaneous.csv")
print(espont.shape)
espont.head(3)

(1840, 6)


,label,text,source_link,news_title,news_summary,news_link
0,ad hominem,Para mi lo que diga este tipejo tiene cero validez.,https://old.meneame.net/story/como-milmillonarios-provocaron-caida-imperio-romano,Cómo los milmillonarios provocaron la caída del Imperio romano,"La causa del hundimiento del Imperio romano de Occidente fue la avaricia de las élites, las luchas intestinas que entablaron entre sí para alcanzar el poder y su creciente desapego del Estado, afi...",https://www.larazon.es/cultura/historia/20221009/w57unlibcneo7bjqbf4tjwn6vq.html?outputType=amp
1,ad hominem,pero era un HDGP o no?,https://old.meneame.net/story/gaddafi-hablando-sobre-invasion-eeuu-irak-asesinato-saddam,Gaddafi hablando sobre la invasión de EEUU a Irak y el asesinato de Saddam Hussein,"3 años antes de su muerte, advirtió a la Liga Árabe: ""Un líder árabe fue ejecutado en la horca. Cualquiera de ustedes podría ser el próximo. ""Somos amigos de Estados Unidos, pero uno de estos días...",https://twitter.com/RoiLopezRivas/status/1583845910043238400?s=20&t=6Ct7F_u-vMpPfZDoNu-o2Q
2,ad hominem,"El hombre, que acumula una veintena de antecedentes, asegura que los agentes fueron los que fueron a buscarlo y que se mearon en la puerta de su casa",https://old.meneame.net/story/mossos-fuera-sevicio-sabado-noche-c-roig-raval-catalan-tuit,MOSSOS FUERA DE SEVICIO el sábado por la noche en la C/Roig del Raval [En catalán el tuit. El vídeo en castellano.],"MOSSOS FUERA DE SEVICIO el sábado por la noche en la C/Roig del Raval. Según los vecinos: Al menos borrachos, robaron 2 patines, amenazaron a los vecinos, rompieron una puerta a patadas, mearon en...",https://twitter.com/SantiAranya/status/1579434149739663361


In [10]:
espont.isna().sum()

label           0
text            0
source_link     0
news_title      0
news_summary    0
news_link       0
dtype: int64

In [11]:
espont["label"].value_counts()

label
non-fallacy               917
ad hominem                208
fallacy of credibility    185
false dilemma             182
fallacy of extension      153
ad populum                 67
faulty generalization      48
deductive fallacy          42
appeal to emotion          38
Name: count, dtype: int64

In [12]:
print("Textos repetidos:", espont["text"].duplicated().sum())

# Longitud en palabras, con falacia (True) y sin ella (False)
espont["es_falacia"] = espont["label"] != "non-fallacy"
contar_palabras(espont["text"]).groupby(espont["es_falacia"]).describe()

Textos repetidos: 0


,count,mean,std,min,25%,50%,75%,max
es_falacia,,,,,,,,
False,917.0,25.960742,11.272749,6.0,18.0,24.0,32.0,73.0
True,923.0,26.764897,15.934450,3.0,16.0,23.0,33.0,129.0


In [13]:
# Las falacias y los textos sin falacia salen de las mismas noticias:
# importa para repartir train/test por noticia.
print("Noticias distintas:", espont["news_link"].nunique())
tipos_por_noticia = espont.groupby("news_link")["es_falacia"].nunique()
print("Noticias con textos con y sin falacia:", (tipos_por_noticia == 2).sum())

Noticias distintas: 826
Noticias con textos con y sin falacia: 462


In [14]:
# Rasgos de texto escrito informal que afectan a la limpieza y a la división en oraciones
textos = espont["text"]
print("Con saltos de línea:", textos.str.contains("\n").sum())
print("Sin signo final (. ! ? …):", (~textos.str.strip().str.contains(r"[.!?…]$")).sum())
print("Empiezan en minúscula:", textos.str.match(r"[a-záéíóúñ]").sum())

Con saltos de línea: 43
Sin signo final (. ! ? …): 230
Empiezan en minúscula: 112


In [15]:
espont[["label", "text"]].sample(8, random_state=3)

,label,text
1179,non-fallacy,La legislación actual impide el uso de hormonas para engordar ganado.
180,ad hominem,Tu aplaude que no das para mas
806,deductive fallacy,"un pais sin creencias y ritos, es un pais sin futuro, porque esa energia nos hace falta para seguir teniendo esperanza."
317,fallacy of credibility,Soy fisico y trabajo con modelos y defininiciones a diario!! Creo que estas predicando en el desierto la verdad! xD. Pero siempre estamos hablando con nosotros mismos!!!
1643,non-fallacy,"Supongo que depende de la comercializadora: El jueves pasado me pasé a la PVPC de Iberdrola (curenergia), y me llevó solamente alrededor de 10 minutos via web; y al dia siguiente (el viernes) ya s..."
818,faulty generalization,"Eso es una foto de un agujero en el uniforme, no es que no crea que no ha ocurrido, más aún viendo cómo les reciben a hostias y cómo hacen devoluciones en caliente, pero es que esa foto no es prue..."
815,faulty generalization,"mi hermana es castellana, lleva mas de 10 años viviendo en guipuzcoa y nunca ha tenido que callarse ni agachar la cabeza ante nadie. De hecho vive tan bien que no tiene ninguna intención de volver."
1015,non-fallacy,"Clara Serra es de las pocas personas en el entorno de Podemos sin una mentalidad dogmática, dispuesta a debatir con personas que no piensan como ella."


## 4. FallacyES-Political

Cada fila es un fragmento falaz de un debate electoral, con su tipo (`category`, a veces varios separados por coma), la fecha del debate (`document`), el orador, su partido y el texto de alrededor (`prev_context`, `foll_context`). **No hay ejemplos sin falacia.**

In [16]:
political = pd.read_csv(CARPETA_POLITICAL / "fallacyES-political.csv")
print(political.shape)
political.head(3)

(1954, 8)


,document,category,text,prev_context,foll_context,speaker_id,speaker_name,speaker_party
0,1993-05-24,False Cause,"Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista Y esa política, que ha sido una política arrogante, es la que ha gene...","Pueden ser también los jóvenes hoy desorientados, desanimados porque no encuentran un empleo, porque el precio de la vivienda es muy alto o simplemente porque no tienen la motivación para abrirse ...","\nPero si esas son las causas, yo quiero apostar en este momento por las soluciones.",SPEAKER_01,José María Aznar,PP
1,1993-05-24,Appeal to Emotion,"Porque efectivamente hoy los españoles tenemos ante nosotros la oportunidad de recuperar esperanza, de recuperar ilusión, de recuperar capacidad de trabajo. Y es por lo que creo que ha llegado la...","Las causas, en mi opinión y en mi juicio, es la política equivocada que en los últimos años ha seguido el gobierno socialista\nY esa política, que ha sido una política arrogante, es la que ha gene...","\nUn gobierno eficaz, compuesto por personas competentes.",SPEAKER_01,José María Aznar,PP
2,1993-05-24,Ad Hominem,"Me permitirá decirle que con el historial de promesas incumplidas que usted tiene encima, de los españoles para volver a tener la sensibilidad necesaria para acercarse a los problemas de los ciu...","Yo quiero recordarle, en definitiva,\nque usted desde el comienzo tuvo un magnífico caudal de entusiasmo ciudadano.\nJamás un gobernante ha tenido, democrático, ha tenido el apoyo que usted ha ten...","\nPor eso hace falta el cambio ahora mismo.\n[CHANGE TO SPEAKER_02]\nEl problema sigue siendo creer o no creer en el país, sin duda alguna.",SPEAKER_01,José María Aznar,PP


In [17]:
political.isna().sum()

document          0
category          0
text              0
prev_context      1
foll_context      5
speaker_id       10
speaker_name     10
speaker_party    10
dtype: int64

In [18]:
# Filas con más de una etiqueta
es_multietiqueta = political["category"].str.contains(",")
print("Filas con más de una etiqueta:", es_multietiqueta.sum())
political.loc[es_multietiqueta, "category"].value_counts()

Filas con más de una etiqueta: 11


category
Appeal to Authority, False Cause       2
Appeal to Fear, Strawman               1
Appeal to Fear, Poisoning the Well     1
Complex Question, False Cause          1
Appeal to Emotion, False Cause         1
Appeal to Emotion, Complex Question    1
False Analogy, False Cause             1
Appeal to Fear, False Dilemma          1
Ad Hominem, False Dilemma              1
Appeal to Emotion, False Dilemma       1
Name: count, dtype: int64

In [19]:
# Recuento por etiqueta (una fila con dos etiquetas cuenta en las dos)
political["category"].str.split(", ").explode().value_counts()

category
Strawman                351
False Cause             304
Ad Hominem              285
False Dilemma           181
Appeal to Emotion       155
Poisoning the Well      152
Appeal to Fear          115
Appeal to Authority      87
Complex Question         74
False Analogy            67
Hasty Generalization     59
Ad Populum               50
Red Herring              28
Slippery Slope           23
Flag Waving              19
Slogan                   15
Name: count, dtype: int64

In [20]:
# Falacias por debate. La fecha (document) identifica cada debate.
debates = pd.read_csv(CARPETA_POLITICAL / "debates_info.csv")
falacias_por_debate = political["document"].value_counts().rename("falacias")
debates.set_index("Date").join(falacias_por_debate).sort_index()

,Participants,Organizer,falacias
Date,,,
1993-05-24,"José María Aznar (PP), Felipe González (PSOE)",Antena 3,88
2008-02-25,"Mariano Rajoy (PP), José Luis R. Zapatero (PSOE)",AcademiaTV,120
2008-03-03,"Mariano Rajoy (PP), José Luis R. Zapatero (PSOE)",AcademiaTV,110
2011-11-07,"Mariano Rajoy (PP), Alfredo Pérez Rubalcaba (PSOE)",AcademiaTV,83
2015-11-23,"Pablo Iglesias (Podemos), Albert Rivera (Cs)",Univ.Carlos III,51
2015-11-30,"Pedro Sánchez (PSOE), Pablo Iglesias (Podemos), Albert Rivera (Cs)",El País,69
2015-12-14,"Pedro Sánchez (PSOE), Mariano Rajoy (PP)",Atresmedia-AcademiaTV,50
2016-06-13,"Mariano Rajoy (PP), Pedro Sánchez (PSOE), Pablo Iglesias (Podemos), Albert Rivera (Cs)",AcademiaTV,89
2019-04-16,"Cayetana Álvarez de Toledo (PP), María Jesús Montero (PSOE), Irene Montero (UP), Inés Arrimadas (Cs), Gabriel Rufián (ERC), Aitor Esteban (PNV)",RTVE,86


In [21]:
political["speaker_party"].value_counts()

speaker_party
PP         591
PSOE       400
Cs         328
VOX        241
Podemos    119
ERC         61
UP          55
UNKNOWN     48
Sumar       43
Junts       32
PNV         18
EHBILDU      8
Name: count, dtype: int64

In [22]:
political["speaker_name"].value_counts()

speaker_name
Mariano Rajoy                    224
Albert Rivera                    212
Pedro Sánchez                    185
Pablo Iglesias                   119
Santiago Abascal                 118
Pablo Casado                     114
José Luis R. Zapatero             84
Inés Arrimadas                    78
Iván Espinosa de los Monteros     72
Gabriel Rufián                    61
Alberto Núñez Feijóo              60
José María Aznar                  57
Cayetana Álvarez de Toledo        55
MODERADOR                         44
Irene Montero                     41
Cuca Gamarra                      37
Alfredo Pérez Rubalcaba           33
Laura Borràs                      32
Rocío Monasterio                  32
Felipe González                   31
Teodoro García Egea               29
Yolanda Díaz                      23
Toni Cantó                        21
María Jesús Montero               20
Aina Vidal                        20
Javier Ortega Smith               19
Aitor Esteban            

In [23]:
# MODERADOR y DECLARACIONES no son candidatos: ¿qué falacias se les atribuyen?
no_candidatos = political[political["speaker_name"].isin(["MODERADOR", "DECLARACIONES"])]
print("Falacias de MODERADOR o DECLARACIONES:", len(no_candidatos))
no_candidatos[["document", "speaker_name", "category", "text"]].head()

Falacias de MODERADOR o DECLARACIONES: 48


,document,speaker_name,category,text
180,2015-11-30,MODERADOR,Appeal to Fear,"Pero antes de comenzar con el bloque de economía, noticia de estas últimas semanas, el terrorismo islamista, algo que nos preocupa mucho y que es verdaderamente grave. Hace unas semanas ha sido e..."
189,2015-11-30,MODERADOR,Appeal to Emotion,"Hoy muchos jóvenes que nos ven saben que aunque trabajen bien, aunque se esfuercen, se les acaba el contrato temporal y les echan y contratan a otro, como si fueran Kleenex. Y yo no quiero jóvene..."
221,2015-12-14,MODERADOR,False Cause,"Absolutamente, pero para eso necesitamos un cambio de gobierno. Porque si algo ha demostrado precisamente este gobierno es que no sé si podremos lograr la recuperación económica con la derecha, ..."
235,2015-12-14,MODERADOR,Red Herring,"La política exterior ya es política nacional. La política europea es política nacional. Lo que pasa, por ejemplo, o lo que ha ocurrido en Kabul, y desde luego todo mi reconocimiento a los policí..."
236,2015-12-14,MODERADOR,Red Herring,Vamos a seguir en este ámbito en el que estamos hablando. ¿Tienen algunos planes para relanzar la economía española también mirando al exterior? Estamos en un ámbito en el que nos conformamos co...


In [24]:
contar_palabras(political["text"]).describe()

count    1954.000000
mean       41.534800
std        38.622228
min         3.000000
25%        20.000000
50%        30.000000
75%        50.000000
max       412.000000
Name: text, dtype: float64

In [25]:
# Restos de la transcripción automática (WhisperX):
# marcas de cambio de orador, p. ej. "[CHANGE TO SPEAKER_05]", y dobles espacios
marca_orador = r"\[CHANGE TO [^\]]*\]"
for columna in ["text", "prev_context", "foll_context"]:
    print(f"{columna}: {political[columna].str.contains(marca_orador).sum()} con marcas de orador")
print("text con dobles espacios:", political["text"].str.contains("  ").sum())

text: 9 con marcas de orador
prev_context: 898 con marcas de orador
foll_context: 885 con marcas de orador
text con dobles espacios: 1054


In [26]:
political[["document", "speaker_name", "category", "text"]].sample(5, random_state=1)

,document,speaker_name,category,text
1812,2023-07-13,Iván Espinosa de los Monteros,Hasty Generalization,"A los españoles les preocupa la educación de sus hijos y les preocupa que cada vez sea peor, que cada vez sea menos exigente, que se haya quebrado lo que era, sin duda, el mejor ascensor social, q..."
1576,2019-11-02,Laura Borràs,Ad Hominem,"Felipe VI, el rey del ""a por ellos"", iba a ser el último rey en abusar de los catalanes, el último borbón en abusar de los catalanes"
1926,2023-07-19,Pedro Sánchez,Poisoning the Well,"no es que le dé vergüenza los votos del señor Abascal, que los asume. Tampoco sea vergüenza de las políticas del señor Abascal, que las asume también. Tampoco sea vergüenza de gobernar con Vox, ..."
1378,2019-04-22,Pablo Casado,False Cause,Y si yo gané las primarias de mi partido fue porque tuve un comportamiento muy firme contra cualquier caso de corrupción.
1446,2019-04-23,Albert Rivera,Flag Waving,que sueñen en la España que queremos. Los españoles sois mi gran familia también. Todos vosotros sois parte de una gran familia que es España.


## 5. Resumen de la exploración

| | Prototípicas | Espontáneas | Political |
|---|---|---|---|
| Filas | 2.083 | 1.840 | 1.954 |
| Clases | 12 | 8 + `non-fallacy` | 16 |
| Sin falacia | 1.318 parejas escritas a mano | 917 textos reales | ninguno |
| Palabras por texto (media) | 25 | 26 | 42 |
| Multietiqueta | no | no | 11 filas |

Lo que habrá que tratar en los siguientes pasos:

- **Prototípicas**: 48 filas tienen un texto que aparece más de una vez (y uno de esos textos tiene dos etiquetas distintas): hay que quitar los duplicados antes de hacer los splits. 765 falacias no tienen pareja sin falacia.
- **Espontáneas**: es texto escrito informal: 43 textos con saltos de línea, 230 sin signo final y 112 que empiezan en minúscula. Afecta a la división en oraciones. Las falacias y los textos sin falacia salen de las mismas noticias (462 de 826), lo que confirma que los splits deben hacerse por noticia.
- **Political**:
  - Restos de la transcripción automática: dobles espacios en 1.054 textos (donde la transcripción no puso punto) y marcas de cambio de orador (`[CHANGE TO SPEAKER_05]`) en 9 textos y en casi la mitad de los contextos.
  - El orador no siempre es fiable: 10 falacias no tienen orador y 48 se atribuyen a `MODERADOR` o `DECLARACIONES`, aunque varias son claramente de un candidato (p. ej. *"necesitamos un cambio de gobierno"*). Parecen errores al separar quién habla en la transcripción. Para el análisis por político habrá que descartarlas o revisarlas.
  - Un nombre viene mal escrito en el original: `Adrana Lastra` (Adriana Lastra).